# 1. Project Title: Handwritten Digit Recognition (ANN vs CNN on MNIST)

**Kit:** AI Practical Project Learning Kit · **Project 4 of 5** · **Domain:** Deep Learning and Computer Vision · **Difficulty:** Intermediate

**GPU:** switch on a free GPU with `Runtime → Change runtime type → T4 GPU`. This project also runs on CPU, only slower (a few minutes). A GPU helps because neural networks are mostly large matrix multiplications, which GPUs do in parallel.

## 2. Project Overview

Neural networks are the engine behind modern AI. In this project you will:

1. See how **one artificial neuron** learns, using only NumPy (forward pass, loss, backpropagation).
2. Build a fully-connected **Artificial Neural Network (ANN)** with Keras.
3. Build a **Convolutional Neural Network (CNN)** that understands images better.
4. Train both on **MNIST** (70,000 images of digits), compare them, and test them on your own handwriting.

## 3. Real-World Problem Statement

Banks read the amount written on cheques, post offices sort mail by handwritten PIN codes, and exam apps digitise handwritten answers. Each of these starts with the same step: **look at a picture of a handwritten character and decide which character it is.**

**Our task:** given a 28×28 grey-scale picture, predict the digit 0–9.

## 4. Project Objectives

1. Explain neurons, weights, biases and activation functions.
2. Explain forward propagation, loss, backpropagation and optimizers.
3. Explain epochs, batch size and learning rate.
4. Train an ANN and a CNN and compare them fairly.
5. Read accuracy/loss curves and a confusion matrix.
6. Predict a custom handwritten digit and show class probabilities.

## 5. What We Will Build

`28×28 image → [ANN or CNN] → 10 probabilities (digits 0–9) → predicted digit`

plus a from-scratch NumPy neuron so you understand what Keras does internally.

## 6. Technologies and Libraries Used

| Library | Purpose |
|---|---|
| `tensorflow` / `keras` | Building and training neural networks (pre-installed in Colab) |
| `numpy` | Arrays and the from-scratch neuron |
| `matplotlib` | Charts and image display |
| `scikit-learn` | Confusion matrix and classification report |
| `Pillow (PIL)` | Loading and resizing your own image |

## 7. System Architecture

```
MNIST (60,000 train + 10,000 test images)
        │  scale pixels 0-255 → 0-1
        ▼
 ┌──────────────┐        ┌──────────────────────────┐
 │ ANN          │        │ CNN                      │
 │ Flatten      │        │ Conv → Pool → Conv → Pool│
 │ Dense(128)   │        │ Flatten → Dense(128)     │
 │ Dense(10)    │        │ Dense(10, softmax)       │
 └──────┬───────┘        └────────────┬─────────────┘
        └──── train (Adam, cross-entropy) ────┘
                      │
   Evaluate on TEST: accuracy · confusion matrix
                      │
   Predict your own digit image ─► probabilities
```

## 8. Understanding the Dataset

**MNIST** (Modified National Institute of Standards and Technology) has **60,000 training** and **10,000 test** images. Each image is **28×28 pixels, grey-scale**, so each pixel is a number from 0 (black) to 255 (white), and shows one digit written by a US Census Bureau employee or a high-school student, **centred, white on black**. The label is the digit 0–9.

Because MNIST is clean and centred, models score very high on it, but they can fail on photos of digits in other styles. We will see that when testing your own images.

## 9. Dataset Download / Loading

In [ ]:
# Section 9 - imports and loading MNIST (built into Keras, ~11 MB)
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

SEED = 42
keras.utils.set_random_seed(SEED)       # seeds Python, NumPy and TensorFlow together

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print("Train images:", x_train.shape, "| Test images:", x_test.shape)
print("Pixel value range:", x_train.min(), "to", x_train.max(), "| dtype:", x_train.dtype)

## 10. Dataset Exploration

In [ ]:
# Section 10a - look at the images
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[i], cmap="gray"); ax.set_title(int(y_train[i])); ax.axis("off")
plt.suptitle("First 20 training images and their labels"); plt.show()

# Section 10b - are the classes balanced?
counts = np.bincount(y_train)
plt.figure(figsize=(7, 3)); plt.bar(range(10), counts, color="steelblue")
plt.xticks(range(10)); plt.title("Training images per digit"); plt.xlabel("digit"); plt.show()
print("Images per digit:", dict(enumerate(counts)))

The classes are roughly balanced (about 6,000 each), so accuracy is a reasonable metric here.

In [ ]:
# Section 10c - what a single image really is: a grid of numbers
print("Label:", y_train[0])
np.set_printoptions(linewidth=200)
print(x_train[0][8:20, 8:20])         # a 12x12 crop of pixel values

# Average image of each digit shows the 'typical shape'
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for d, ax in enumerate(axes):
    ax.imshow(x_train[y_train == d].mean(axis=0), cmap="gray"); ax.set_title(d); ax.axis("off")
plt.suptitle("Average image per digit"); plt.show()

## 11. Data Preprocessing

1. **Normalize**: divide pixels by 255 so values lie in 0–1. Neural networks train faster and more stably on small numbers.
2. **Reshape for the CNN**: a CNN expects a channel dimension, so `(28, 28)` becomes `(28, 28, 1)`. (1 channel = grey-scale; colour images have 3.)
3. **Labels** stay as integers 0–9. We use `sparse_categorical_crossentropy`, which accepts integer labels directly.
4. **Validation split**: Keras sets aside 10 % of the training images (`validation_split=0.1`) to monitor the model while training, while the 10,000 test images remain untouched until the end.

In [ ]:
# Section 11 - normalize and reshape
x_train_n = (x_train.astype("float32") / 255.0)[..., np.newaxis]   # shape (60000, 28, 28, 1)
x_test_n = (x_test.astype("float32") / 255.0)[..., np.newaxis]
print("Shape for CNN:", x_train_n.shape, "| value range:", x_train_n.min(), "to", x_train_n.max())

## 12. Understanding the Algorithm

### An artificial neuron
`output = activation(w1·x1 + w2·x2 + ... + b)`
* **Weights (w)** say how important each input is; **bias (b)** shifts the result.
* The **activation function** adds non-linearity so stacked layers can learn curves, not only straight lines. **ReLU** = `max(0, x)` is the standard for hidden layers. **Softmax** turns 10 raw scores into 10 probabilities that sum to 1 (used in the last layer).

### Learning = forward pass + loss + backpropagation + optimizer
1. **Forward propagation:** inputs flow through the layers to produce a prediction.
2. **Loss function:** measures how wrong the prediction is. For classification we use **cross-entropy**: it is small when the correct class gets a high probability, and huge when it gets a low one.
3. **Backpropagation:** uses the chain rule from calculus to compute how much *each weight* contributed to the loss (the **gradient**).
4. **Optimizer:** nudges each weight a small step against its gradient. **Adam** is a popular choice that adapts the step size for each weight. The step size is the **learning rate**.

### Epochs and batch size
* **Batch size** = how many images are processed before the weights are updated (128 here).
* **Epoch** = one full pass over all training images. 60,000 images ÷ 128 ≈ 422 updates per epoch.

### CNN vs ANN
An **ANN** flattens the picture into 784 numbers and loses the information about which pixels are neighbours. A **CNN** slides small **filters** (3×3 windows) across the image to detect edges and strokes; **pooling** shrinks the image while keeping the strongest signals. The same filter is reused everywhere, so a CNN needs far fewer weights and recognises a shape wherever it appears.

### Hands-on: one neuron learns in pure NumPy

Before using Keras, we make one neuron learn the **AND** logic gate with forward propagation, cross-entropy loss and backpropagation written by hand. This is exactly what Keras automates.

In [ ]:
# Section 12 - a single neuron trained from scratch with NumPy
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_and = np.array([0, 0, 0, 1], dtype=float)

rng = np.random.default_rng(SEED)
w = rng.normal(size=2)          # weights
b = 0.0                         # bias
lr = 0.5                        # learning rate

def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

history_loss = []
for epoch in range(1, 2001):
    z = X_and @ w + b                       # forward: weighted sum
    p = sigmoid(z)                          # forward: activation
    loss = -np.mean(y_and * np.log(p + 1e-9) + (1 - y_and) * np.log(1 - p + 1e-9))   # cross-entropy
    grad_z = (p - y_and) / len(y_and)       # backprop: dLoss/dz for sigmoid + cross-entropy
    w -= lr * (X_and.T @ grad_z)            # backprop: dLoss/dw, then take a step
    b -= lr * grad_z.sum()                  # backprop: dLoss/db
    history_loss.append(loss)
    if epoch in (1, 10, 100, 500, 2000):
        print(f"epoch {epoch:5d}  loss = {loss:.4f}")

print("\nLearned weights:", w.round(2), "bias:", round(b, 2))
for x_row, prob in zip(X_and, sigmoid(X_and @ w + b)):
    print(f"  {x_row.astype(int)} -> probability of 1 = {prob:.3f}  -> predicts {int(prob > 0.5)}")
plt.figure(figsize=(6, 3)); plt.plot(history_loss); plt.title("Loss falls as the neuron learns"); plt.xlabel("epoch"); plt.show()

The loss goes down because each backpropagation step moves the weights in the direction that reduces the error. The big networks below do the same thing, only with hundreds of thousands of weights.

## 13. Building the Model

We build two networks. `summary()` shows each layer and its number of parameters (weights + biases). Compare the totals.

In [ ]:
# Section 13 - ANN and CNN definitions
def build_ann():
    return keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Flatten(),                         # 28x28 grid -> 784 numbers
        layers.Dense(128, activation="relu"),     # 128 neurons
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),   # 10 probabilities
    ], name="ANN")

def build_cnn():
    return keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation="relu"),   # 32 filters detect edges/strokes
        layers.MaxPooling2D((2, 2)),                    # keep strongest signals, halve the size
        layers.Conv2D(64, (3, 3), activation="relu"),   # filters detect combinations of strokes
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),                            # randomly switch off 30% of neurons while training (reduces overfitting)
        layers.Dense(10, activation="softmax"),
    ], name="CNN")

ann, cnn = build_ann(), build_cnn()
ann.summary()
cnn.summary()
print("ANN parameters:", ann.count_params(), "| CNN parameters:", cnn.count_params())

**Compiling** chooses how to learn: `optimizer="adam"`, `loss="sparse_categorical_crossentropy"` (cross-entropy for integer labels) and `metrics=["accuracy"]` (what we want to watch).

In [ ]:
# Section 13b - compile both models
for m in (ann, cnn):
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("Compiled:", ann.name, "and", cnn.name)

## 14. Training the Model

`fit()` runs the epochs. `EarlyStopping` stops training when validation loss stops improving and restores the best weights, a guard against overfitting. We allow up to 10 epochs.

In [ ]:
# Section 14 - train both models (watch accuracy rise and loss fall each epoch)
early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
EPOCHS, BATCH = 10, 128

print("Training the ANN ...")
hist_ann = ann.fit(x_train_n, y_train, validation_split=0.1, epochs=EPOCHS, batch_size=BATCH,
                   callbacks=[early], verbose=2)
print("\nTraining the CNN ...")
hist_cnn = cnn.fit(x_train_n, y_train, validation_split=0.1, epochs=EPOCHS, batch_size=BATCH,
                   callbacks=[early], verbose=2)

## 15. Model Evaluation

Finally we use the **10,000 test images** the models have never seen.

In [ ]:
# Section 15 - evaluate on the test set
ann_loss, ann_acc = ann.evaluate(x_test_n, y_test, verbose=0)
cnn_loss, cnn_acc = cnn.evaluate(x_test_n, y_test, verbose=0)
print(f"ANN  test accuracy: {ann_acc:.4f}   test loss: {ann_loss:.4f}   parameters: {ann.count_params():,}")
print(f"CNN  test accuracy: {cnn_acc:.4f}   test loss: {cnn_loss:.4f}   parameters: {cnn.count_params():,}")

probs_ann = ann.predict(x_test_n, verbose=0); pred_ann = probs_ann.argmax(axis=1)
probs_cnn = cnn.predict(x_test_n, verbose=0); pred_cnn = probs_cnn.argmax(axis=1)
print("\nCNN classification report:")
print(classification_report(y_test, pred_cnn, digits=4))

**Reading it:** *precision* and *recall* are given per digit. The lowest rows show the digits the CNN finds hardest. Your values come from your own run and will vary slightly between runs and hardware.

## 16. Data Visualization

In [ ]:
# Section 16a - accuracy and loss curves for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, h, style in [("ANN", hist_ann, "--"), ("CNN", hist_cnn, "-")]:
    axes[0].plot(h.history["accuracy"], style, label=f"{name} train")
    axes[0].plot(h.history["val_accuracy"], style, label=f"{name} validation")
    axes[1].plot(h.history["loss"], style, label=f"{name} train")
    axes[1].plot(h.history["val_loss"], style, label=f"{name} validation")
axes[0].set_title("Accuracy"); axes[1].set_title("Loss")
for ax in axes: ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout(); plt.show()

A big gap between the *train* and *validation* lines means **overfitting**: the model memorises training images. If both lines are low, the model is **underfitting**.

In [ ]:
# Section 16b - confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, pred, name in zip(axes, [pred_ann, pred_cnn], ["ANN", "CNN"]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=range(10)).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name} confusion matrix (rows = true digit, columns = predicted)")
plt.tight_layout(); plt.show()

In [ ]:
# Section 16c - ANN vs CNN comparison and the CNN's mistakes
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["ANN", "CNN"], [ann_acc, cnn_acc], color=["#dd8452", "#4c72b0"])
ax.set_ylim(min(ann_acc, cnn_acc) - 0.02, 1.0); ax.set_title("Test accuracy"); plt.show()

wrong = np.where(pred_cnn != y_test)[0]
print("CNN mistakes on the test set:", len(wrong), "of", len(y_test))
fig, axes = plt.subplots(2, 8, figsize=(15, 4.5))
for ax, i in zip(axes.ravel(), wrong[:16]):
    ax.imshow(x_test[i], cmap="gray"); ax.axis("off")
    ax.set_title(f"true {y_test[i]}\npred {pred_cnn[i]} ({probs_cnn[i].max():.0%})", fontsize=9)
plt.suptitle("Some digits the CNN got wrong - many are hard even for people"); plt.show()

## 17. Testing with New Inputs

First we predict one test image and show all 10 probabilities. Then you can **upload your own** handwritten digit.

In [ ]:
# Section 17a - probabilities for one test image
def show_prediction(img28, model, title=""):
    # img28: 28x28 array with values 0-1 (white digit on black)
    p = model.predict(img28.reshape(1, 28, 28, 1), verbose=0)[0]
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    axes[0].imshow(img28, cmap="gray"); axes[0].axis("off"); axes[0].set_title(title or "Input image")
    axes[1].bar(range(10), p, color="seagreen"); axes[1].set_xticks(range(10))
    axes[1].set_ylim(0, 1); axes[1].set_xlabel("digit"); axes[1].set_ylabel("probability")
    axes[1].set_title(f"Prediction: {p.argmax()}  ({p.max():.1%})")
    plt.tight_layout(); plt.show()
    return p

idx = 7
_ = show_prediction(x_test_n[idx, :, :, 0], cnn, f"Test image #{idx} (true label {y_test[idx]})")

### Your own handwriting
Write one digit **thick and dark on white paper** (a marker works best), take a photo or scan, and upload it. Our function converts it to grey-scale, **inverts it if the background is light** (MNIST is white-on-black), crops to the digit, pads it to a square, resizes to 28×28 and normalizes. If you do not upload anything, the cell uses a saved test image so that it still runs.

In [ ]:
# Section 17b - preprocess and predict a custom image
from PIL import Image, ImageOps

def preprocess_digit(path):
    img = Image.open(path).convert("L")                     # grey-scale
    arr = np.asarray(img, dtype="float32")
    if arr.mean() > 127:                                    # light background -> invert
        arr = 255 - arr
    ys, xs = np.where(arr > 0.3 * arr.max())                # find the digit's bounding box
    if len(ys) == 0:
        raise ValueError("No digit found - is the image blank?")
    arr = arr[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    side = max(arr.shape)
    canvas = np.zeros((side, side), dtype="float32")        # pad to square, centred
    r0, c0 = (side - arr.shape[0]) // 2, (side - arr.shape[1]) // 2
    canvas[r0:r0 + arr.shape[0], c0:c0 + arr.shape[1]] = arr
    small = Image.fromarray(canvas.astype("uint8")).resize((20, 20))     # MNIST digits fill a 20x20 box
    out = np.zeros((28, 28), dtype="float32")
    out[4:24, 4:24] = np.asarray(small, dtype="float32")
    return out / 255.0

path = None
try:
    from google.colab import files
    print("Choose an image file of ONE handwritten digit (or cancel to use a sample).")
    up = files.upload()
    if up:
        path = next(iter(up))
except Exception as e:                     # not in Colab, or the user cancelled
    print("No upload available:", type(e).__name__)

if path is None:
    Image.fromarray(255 - x_test[idx]).save("sample_digit.png")   # a saved black-on-white sample
    path = "sample_digit.png"
    print("Using the sample image", path)

try:
    digit = preprocess_digit(path)
    p_cnn = show_prediction(digit, cnn, "Your digit after preprocessing (CNN)")
    p_ann = ann.predict(digit.reshape(1, 28, 28, 1), verbose=0)[0]
    print(f"CNN says {p_cnn.argmax()} ({p_cnn.max():.1%}) | ANN says {p_ann.argmax()} ({p_ann.max():.1%})")
except Exception as e:
    print("Could not process the image:", e)

If a high-confidence answer is wrong, that is normal: MNIST models only know the style of MNIST. Thin pen strokes, tilted or off-centre digits and busy backgrounds all reduce accuracy.

## 18. Saving the Trained Model

In [ ]:
# Section 18 - save, reload and verify the CNN
cnn.save("mnist_cnn.keras")
reloaded = keras.models.load_model("mnist_cnn.keras")
same = np.allclose(reloaded.predict(x_test_n[:100], verbose=0), cnn.predict(x_test_n[:100], verbose=0), atol=1e-5)
print("Reloaded model gives the same predictions:", same)
try:
    from google.colab import files
    files.download("mnist_cnn.keras")
except ImportError:
    print("(Not running in Colab - the file is in the current folder.)")

## 19. Final Output

In [ ]:
# Section 19 - final summary
print("=" * 60); print("HANDWRITTEN DIGIT RECOGNITION - FINAL SUMMARY"); print("=" * 60)
print(f"{'Model':6s} {'Test accuracy':>14s} {'Parameters':>12s} {'Epochs run':>11s}")
print(f"{'ANN':6s} {ann_acc:14.4f} {ann.count_params():12,d} {len(hist_ann.history['loss']):11d}")
print(f"{'CNN':6s} {cnn_acc:14.4f} {cnn.count_params():12,d} {len(hist_cnn.history['loss']):11d}")
print("Better on the test set:", "CNN" if cnn_acc >= ann_acc else "ANN")

## 20. How the Complete Project Works

1. MNIST images are loaded and pixel values scaled to 0–1.
2. A NumPy neuron shows forward propagation, loss and backpropagation by hand.
3. Keras builds an ANN (flatten + dense layers) and a CNN (convolution + pooling + dense layers).
4. Both are trained with Adam and cross-entropy, watching a validation split, with early stopping.
5. The 10,000 test images measure true performance; the confusion matrix shows which digits are mixed up.
6. Your own image is cleaned (inverted, cropped, centred, resized) and the model outputs 10 probabilities.
7. The model is saved as a `.keras` file for reuse.

## 21. Limitations

* MNIST digits are clean, centred and written in one style; real-world handwriting, photos and other scripts are much harder.
* The model always outputs *a* digit, even for a letter or a scribble. It has no "I don't know" option.
* High confidence does not guarantee correctness.
* Results vary slightly from run to run, even with seeds, because GPU maths is not always deterministic.

## 22. Real-World Applications

Cheque and form reading · postal code sorting · digitising exam sheets · number-plate and meter reading · the building block for general **OCR** and for any image-classification task.

## 23. Future Enhancements

* **Data augmentation** (small rotations, shifts, zooms) to make the model tougher on messy digits.
* Train on **EMNIST** to recognise letters as well.
* Add a drawing canvas in a Gradio or Streamlit app for live predictions.
* Add an "unsure" threshold: refuse to answer when the highest probability is below, for example, 0.8.
* Try deeper CNNs, batch normalisation, and learning-rate schedules.

## 24. Student Mini Challenges

1. Change the ANN to have 512 neurons in the first layer. Does accuracy improve? Does it overfit more?
2. Change `batch_size` to 32 and 512. What changes in speed and accuracy?
3. Replace `adam` with `keras.optimizers.SGD(learning_rate=0.01)`. How does the loss curve change?
4. Remove `Dropout` from the CNN and compare train vs validation accuracy.
5. Add `RandomRotation(0.1)` and `RandomTranslation(0.1, 0.1)` layers and test on tilted digits.
6. Upload ten of your own digits and compute how many the CNN gets right.
7. In the NumPy neuron, change the target to the OR gate. Can the same neuron learn XOR? Explain why not.